# Preparação reproduzível — SEEG 10 e IBGE

Gera todos os CSVs consumidos pelo notebook 02. Execute com o ambiente descrito no README, da raiz ou de `notebook/`. Os dados cobrem 1970–2021; não são uma atualização para o ano corrente.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/ghg.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.ghg import prepare, validate_processed, SECTOR
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
IMAGES_DIR = PROJECT_ROOT / "images"
IMAGES_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

## Método e cobertura

Usamos apenas a categoria `Emissão` e `CO2e (t) GWP-AR5`, sem remoções, NCI ou bunkers. O total nacional inclui registros sem UF; rankings estaduais contêm apenas as 27 UFs. O código rejeita agregados `BR` para impedir dupla contagem. Pequenos valores negativos já classificados como emissão no arquivo original são preservados, sem reclassificação arbitrária.

Mudança de Uso da Terra e Floresta começa em 1990: a série completa só é comparável desde esse ano. Na versão inicial, a população de 2022 é uma aproximação explícita para emissões de 2021.

In [2]:
outputs = prepare(PROJECT_ROOT)
display(pd.DataFrame({"arquivo": outputs.keys(), "linhas": [len(df) for df in outputs.values()]}))

,arquivo,linhas
0,emissoes_co2e_gwp_ar5_1970_2021.csv,445640
1,emissao_anual_brasil.csv,52
2,emissao_estado_2021.csv,27
3,emissao_setor_2021.csv,5
4,emissao_per_capita_2021.csv,27
5,alocacao_emissoes_2021.csv,2


## Verificações e reconciliação

Os controles conferem cobertura, unicidade das UFs, correspondência com a população, fórmulas e totais. O total nacional é igual à soma das UFs **mais** a parcela sem alocação estadual.

In [3]:
checks = validate_processed(PROJECT_ROOT)
display(pd.Series(checks))
display(outputs["alocacao_emissoes_2021.csv"])

ufs                    2.700000e+01
anos                   5.200000e+01
nacional_2021_tco2e    2.422625e+09
alocado_ufs_tco2e      2.337048e+09
dtype: float64

,Alocacao,Emissao
0,Alocado às UFs,2.337048e+09
1,Sem alocação estadual,8.557690e+07


## Amostras das saídas

Os nomes e as colunas abaixo constituem o contrato de entrada do notebook exploratório.

In [4]:
for name, frame in outputs.items():
    print(name)
    display(frame.head(3))

emissoes_co2e_gwp_ar5_1970_2021.csv


,Nível 1 - Setor,Nível 2,Nível 3,Nível 4,Nível 5,Nível 6,Gás,Estado,Atividade Econômica,Produto,Ano,Emissao
0,Processos Industriais,Indústria Química,Produção de Ácido Adípico,NaN,NaN,NaN,CO2e (t) GWP-AR5,SP,OUTRA_IND,NaN,1970,1.645650e+06
1,Processos Industriais,Indústria Química,Produção de Ácido Nítrico,NaN,NaN,NaN,CO2e (t) GWP-AR5,BA,OUTRA_IND,NaN,1970,0.000000e+00
2,Processos Industriais,Indústria Química,Produção de Ácido Nítrico,NaN,NaN,NaN,CO2e (t) GWP-AR5,RJ,OUTRA_IND,NaN,1970,2.344758e+02


emissao_anual_brasil.csv


,Ano,Emissao
0,1970,3.422406e+08
1,1971,3.622887e+08
2,1972,3.827717e+08


emissao_estado_2021.csv


,Estado,Emissao
13,PA,4.479274e+08
12,MT,2.693745e+08
10,MG,1.682337e+08


... (2 outputs left out)

emissao_per_capita_2021.csv


,Estado,Emissao,UF,populacao,emissao_per_capita
13,RR,6.157727e+07,RR,634805,97.001873
5,RO,1.378898e+08,RO,1616379,85.307831
1,MT,2.693745e+08,MT,3784239,71.183268


alocacao_emissoes_2021.csv


,Alocacao,Emissao
0,Alocado às UFs,2.337048e+09
1,Sem alocação estadual,8.557690e+07


## Fontes

- SEEG 10: `data/raw/1-SEEG10_GERAL-BR_UF_2022.10.27-FINAL-SITE.xlsx`, aba `GEE Estados`.
- IBGE: `data/raw/POP2022_Municipios.xls`.
- Transformações e regras: `src/ghg.py`.

Continue em `02_exploratory_analysis.ipynb`.